In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/dataset_wav16k

/content/drive/MyDrive/Colab Notebooks/dataset_wav16k


In [3]:
!pip -q install openai sentence-transformers openpyxl xlrd==2.0.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.8 MB/s eta 0:00:00


In [4]:
import re
import os
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import openpyxl
import xlrd

BASE_DIR = "/content/"
# (опционально) семантический скоринг кандидатов
USE_EMBEDDINGS = True
if USE_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as e:
        print(f"Warning: Could not import SentenceTransformer. Semantic scoring will be disabled. Error: {e}")
        USE_EMBEDDINGS = False

# --- Пути к вашим файлам (в Colab положите в /content или подключите Google Drive) ---
STEMS_XLSX = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_stems_unik_edit.xlsx"
ENDINGS_XLS = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_endings_morph_5994.xls"
STOPWORDS_TXT = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/stop_words.txt"

In [5]:
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_keep_punct(text: str) -> List[str]:
    return WORD_RE.findall(text)

def is_word(token: str) -> bool:
    return token.isalpha()

def normalize(token: str) -> str:
    # Важно: не ломаем казахские буквы. Просто lower().
    return token.lower()

def levenshtein(a: str, b: str, max_dist: int = 2) -> int:
    """
    Быстрый Левенштейн с отсечкой: если > max_dist — возвращаем max_dist+1
    """
    if a == b:
        return 0
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1
    # DP по строке
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        min_row = cur[0]
        for j, cb in enumerate(b, start=1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (ca != cb)
            v = min(ins, dele, sub)
            cur.append(v)
            if v < min_row:
                min_row = v
        prev = cur
        if min_row > max_dist:
            return max_dist + 1
    return prev[-1]


In [6]:
def load_stopwords(path: str) -> set:
    sw = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            w = line.strip()
            if w:
                sw.add(w.lower())
    return sw

def load_stems_xlsx(path: str) -> List[str]:
    wb = openpyxl.load_workbook(path)
    sh = wb.active
    stems = []
    for row in sh.iter_rows(values_only=True):
        if not row:
            continue
        stem = row[0]
        if stem is None:
            continue
        stem = str(stem).replace("\ufeff", "").strip()
        if stem:
            stems.append(stem)
    # как у вас: сортировка по длине (самые длинные — первыми)
    stems = sorted(set(stems), key=len, reverse=True)
    return stems

def load_endings_xls(path: str) -> Tuple[List[str], List[str]]:
    """
    Ожидаем:
      col0 = ending
      col1 = morph description (MD)
    как в вашем скрипте сегментации.
    """
    wb = xlrd.open_workbook(path)
    sh = wb.sheet_by_index(0)
    endings = []
    endings_md = []
    for r in range(1, sh.nrows):
        e = sh.cell_value(r, 0)
        md = sh.cell_value(r, 1) if sh.ncols > 1 else ""
        e = str(e).replace("\ufeff", "").strip()
        md = str(md).replace("\ufeff", "").strip()
        endings.append(e)
        endings_md.append(md)
    return endings, endings_md

stop_words = load_stopwords(STOPWORDS_TXT)
stems_list = load_stems_xlsx(STEMS_XLSX)
endings_list, endings_md_list = load_endings_xls(ENDINGS_XLS)

# Быстрые структуры:
stems_set = set([s.lower() for s in stems_list])
ending_to_md = {endings_list[i]: endings_md_list[i] for i in range(len(endings_list))}
endings_set = set(endings_list)

print("Loaded:")
print("  stems:", len(stems_list))
print("  endings:", len(endings_list))
print("  stopwords:", len(stop_words))


Loaded:
  stems: 103615
  endings: 5994
  stopwords: 188


In [7]:
@dataclass
class Analysis:
    ok: bool
    stem: str
    ending: str
    md: str

def split_stem_ending(word: str, min_stem_len: int = 2) -> Tuple[str, str]:
    w = word
    wl = len(w)
    if wl <= min_stem_len:
        return w, ""

    wlow = w.lower()
    if wlow in stems_set:
        return w, ""

    # перебор возможных окончаний: от коротких к длинным/или наоборот
    # у вас: i от n+1 вниз (т.е. пробуем разные длины ending)
    # сделаем: пробуем ending от 1..(wl-min_stem_len), но эффективнее — от длинного к короткому
    max_ending_len = wl - min_stem_len
    for elen in range(max_ending_len, 0, -1):
        ending = w[wl-elen:]
        stem = w[:wl-elen]
        if ending in endings_set and stem.lower() in stems_set:
            return stem, ending

    # если не нашли, возвращаем "как есть"
    return w, ""

def cse_analyze_word(word: str) -> Analysis:
    stem, ending = split_stem_ending(word)
    if ending:
        md = ending_to_md.get(ending, "")
        return Analysis(ok=True, stem=stem, ending=ending, md=md)
    # ok=True если хотя бы stem — известный
    if stem.lower() in stems_set:
        return Analysis(ok=True, stem=stem, ending="", md="")
    return Analysis(ok=False, stem=stem, ending="", md="")


In [8]:
PHONETIC_PAIRS = [
    ("қ","к"), ("ғ","г"),
    ("ұ","у"), ("ү","у"),
    ("ы","і"), ("о","ө")
]
FUNCTION_WORDS = {
    "бірақ","және","немесе","ал","өйткені","сондықтан","егер","онда",
    "мен","сен","ол","біз","сіз","олар",
    "да","де","та","те",
    "емес","екен","еді"
}
ASR_NORMALIZE_MAP = {
    "биз": "біз",
    "бирак": "бірақ",
    "казир": "қазір",
    "себеби": "себебі",
    "жане": "және",
    "озимди": "өзімді",
    "оз": "өз",
    "бугин": "бүгін",
    "озимди": "өзімді",
}


def normalize_asr_token(w: str) -> str:
    wl = w.lower()
    return ASR_NORMALIZE_MAP.get(wl, wl)

def phonetic_variants(word: str) -> List[str]:
    cands = set()
    for a, b in PHONETIC_PAIRS:
        if a in word:
            cands.add(word.replace(a, b))
        if b in word:
            cands.add(word.replace(b, a))
    return list(cands)

def repair_by_endings(word: str, max_ending_dist: int = 1) -> List[str]:
    """
    Если word не анализируется, пробуем:
      - найти разбиение на stem + ending'
      - stem должен быть в stems
      - ending' подбираем как ближайшее к хвосту word по Левенштейну среди endings
    """

    w = word
    wl = len(w)

    # ЖЁСТКО: короткие слова не режем на stem+suffix
    if wl <= 4:
        return []

    out = set()

    # перебираем позицию разделения: stem | suffix
    for cut in range(3, wl-1):  # stem>=3, suffix>=2
        stem = w[:cut]
        if stem.lower() not in stems_set:
            continue

        suffix = w[cut:]
        # 1) если suffix точно является окончанием — уже хорошо
        if suffix in endings_set:
            out.add(stem + suffix)
            continue

        # 2) иначе ищем ближайшее окончание
        best = None
        best_d = max_ending_dist + 1
        for e in endings_list:
            # быстрые фильтры по длине
            if abs(len(e) - len(suffix)) > max_ending_dist:
                continue
            d = levenshtein(suffix, e, max_dist=max_ending_dist)
            if d < best_d:
                best_d = d
                best = e
                if best_d == 0:
                    break
        if best is not None and best_d <= max_ending_dist:
            out.add(stem + best)

    return list(out)

def generate_candidates(word: str) -> List[str]:
    cands = set()
    for v in phonetic_variants(word):
        cands.add(v)
        for r in repair_by_endings(v, max_ending_dist=1):
            cands.add(r)
    for r in repair_by_endings(word, max_ending_dist=1):
        cands.add(r)

    # всегда можно оставить исходное
    cands.add(word)
    return list(cands)


In [9]:
if USE_EMBEDDINGS:
    emb_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a) + 1e-9
    nb = np.linalg.norm(b) + 1e-9
    return float(np.dot(a, b) / (na * nb))

def pick_best_candidate(word: str, candidates: List[str], left_ctx: str, right_ctx: str) -> str:
    """
    left_ctx/right_ctx — слова вокруг (для контекста).
    """
    # 1) фильтр: оставляем анализируемые
    analyzed = []
    for c in candidates:
        a = cse_analyze_word(c)
        if a.ok:
            analyzed.append(c)

    pool = analyzed if analyzed else candidates

    # 2) базовый скоринг: меньше правок => лучше
    base_scores = []
    for c in pool:
        d = levenshtein(word, c, max_dist=3)
        # чем меньше dist, тем лучше
        base = -float(d)
        base_scores.append((c, base))

    # 3) embeddings (опционально): сравниваем фразы "лево + слово + право"
    if USE_EMBEDDINGS and len(pool) > 1:
        sent0 = f"{left_ctx} {word} {right_ctx}".strip()
        v0 = emb_model.encode([sent0])[0]

        best_c = None
        best_score = -1e9
        for c, base in base_scores:
            sent = f"{left_ctx} {c} {right_ctx}".strip()
            v = emb_model.encode([sent])[0]
            sem = cosine(v0, v)
            score = base + 0.5 * sem  # вес семантики (можете менять)
            if score > best_score:
                best_score = score
                best_c = c
        return best_c if best_c is not None else pool[0]

    # без embeddings
    base_scores.sort(key=lambda x: x[1], reverse=True)
    return base_scores[0][0]

def try_fix_word_safe(word: str, left_ctx: str = "", right_ctx: str = "") -> str:
    w = word.lower()

    # Нормализация частых ASR-форм
    w_norm = ASR_NORMALIZE_MAP.get(w, w)
    if w_norm != w:
        return w_norm

    # 1) Короткие слова НЕ исправляем (чтобы не было "Биа", "себеба")
    if len(w) <= 3:
        return word

    # 0) служебные слова не трогаем
    if w in FUNCTION_WORDS:
        return word

    # 1) если слово уже анализируется CSE — не трогаем
    an0 = cse_analyze_word(w)
    if an0.ok:
        return word

    # 2) кандидаты
    cands = generate_candidates(w)

    # 3) первичные фильтры (чтобы не было мусора)
    cands = [c for c in cands if c.isalpha() and 2 <= len(c) <= 25]
    if not cands:
        return word

    # 4) специальный частый хак для ASR: бирақ→бірақ (можно расширять список)
    # делаем ДО общего ранжирования, но только если кандидат есть и он "валиден"
    if levenshtein(w, "бірақ", max_dist=2) <= 1:
        return "бірақ"

    # 5) ЖЁСТКИЙ фильтр: оставляем только кандидаты, которые подтверждаются CSE
    cse_ok = []
    for c in cands:
        a = cse_analyze_word(c)
        if a.ok:
            cse_ok.append(c)

    # Если есть хотя бы один CSE-валидный кандидат — выбираем только среди них
    pool = cse_ok if cse_ok else cands

    # 6) Доп. приоритет: кандидат, который является стемом (без окончания) тоже хороший
    # (это уменьшает странные исправления)
    def priority_score(c: str) -> float:
        base = -float(levenshtein(w, c, max_dist=3))  # меньше правок лучше
        a = cse_analyze_word(c)
        # бонус за CSE-ok
        if a.ok:
            base += 1.0
        # бонус если это "чистый стем"
        if c.lower() in stems_set:
            base += 0.5
        return base

    # 7) Если embeddings выключены — выберем по приоритету
    if not USE_EMBEDDINGS or len(pool) == 1:
        return sorted(pool, key=priority_score, reverse=True)[0]

    # 8) Если embeddings включены — сравниваем контекстные фразы
    sent0 = f"{left_ctx} {w} {right_ctx}".strip()
    v0 = emb_model.encode([sent0])[0]

    best_c = None
    best_score = -1e9

    for c in pool:
        sent = f"{left_ctx} {c} {right_ctx}".strip()
        v = emb_model.encode([sent])[0]
        sem = cosine(v0, v)

        score = priority_score(c) + 0.5 * sem   # 0.5 можно менять
        if score > best_score:
            best_score = score
            best_c = c

    return best_c if best_c is not None else word






/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
def mapp_asr_process(text: str) -> str:
    tokens = tokenize_keep_punct(text)
    out = []

    # для контекста берём соседние токены-слова
    word_positions = [i for i,t in enumerate(tokens) if is_word(t)]
    word_set = set(word_positions)

    for i, tok in enumerate(tokens):
        tlow = normalize(tok)

        # стоп-слова как есть
        if tlow in stop_words:
            out.append(tok)
            continue

        # не слово (пунктуация/числа) — как есть
        if not is_word(tok):
            out.append(tok)
            continue

        # если анализируется — оставляем
        an = cse_analyze_word(tlow)
        if an.ok:
            out.append(tok)
            continue

        # иначе: генерация кандидатов + выбор
        cands = generate_candidates(tlow)

        # простой контекст: ближайшее слово слева/справа
        left = ""
        right = ""
        # слева
        j = i-1
        while j >= 0:
            if j in word_set:
                left = normalize(tokens[j])
                break
            j -= 1
        # справа
        j = i+1
        while j < len(tokens):
            if j in word_set:
                right = normalize(tokens[j])
                break
            j += 1

        #best = pick_best_candidate(tlow, cands, left, right)
        best = try_fix_word_safe(tlow, left, right)

        # сохраняем исходный регистр (минимально): если токен был с заглавной
        if tok[:1].isupper():
            best = best[:1].upper() + best[1:]
        out.append(best)

    # аккуратно склеим: чтобы не было пробелов перед пунктуацией
    # 1) сначала join с пробелами
    joined = " ".join(out)
    # 2) уберём пробелы перед знаками
    joined = re.sub(r"\s+([.,!?;:%)\]\}»])", r"\1", joined)
    joined = re.sub(r"([(\[\{«])\s+", r"\1", joined)
    return joined


In [11]:
from openai import OpenAI

def transcribe_whisper(audio_path: str, api_key: str) -> str:
    client = OpenAI(api_key=api_key)
    with open(audio_path, "rb") as f:
        result = client.audio.transcriptions.create(
            file=f,
            model="whisper-large-v3-turbo"
        )
    return result.text

def transcribe_and_mapp(audio_path: str, api_key: str) -> Tuple[str, str]:
    raw = transcribe_whisper(audio_path, api_key=api_key)
    fixed = mapp_asr_process(raw)
    return raw, fixed


In [12]:
# Тест без аудио:
text = "Биз бирак казир  оқимаймыз, себеби ауа райы жаман."
print("RAW:  ", text)
print("MAPP: ", mapp_asr_process(text))


RAW:   Биз бирак казир  оқимаймыз, себеби ауа райы жаман.
MAPP:  Біз бірақ қазір оқимаймыз, себебі ауа райы жаман.


In [13]:
!pip install -q openai-whisper jiwer sacrebleu bert-score
!apt-get -y -qq install ffmpeg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.5 MB/s eta 0:00:00


In [14]:
AUDIO_MP3   = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/1m/1m_001.wav"
STEMS_XLSX = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_stems_unik_edit.xlsx"
ENDINGS_XLS = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_endings_morph_5994.xls"
STOPWORDS_TXT = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/stop_words.txt"

import os
print("audio exists:", os.path.exists(AUDIO_MP3))
print("stems exists:", os.path.exists(STEMS_XLSX))
print("endings exists:", os.path.exists(ENDINGS_XLS))
print("stopwords exists:", os.path.exists(STOPWORDS_TXT))


audio exists: True
stems exists: True
endings exists: True
stopwords exists: True


In [15]:
import whisper

whisper_model = whisper.load_model("medium")  # немесе "large"

def transcribe_whisper_local(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path, language="kk")
    return result["text"]

raw_text = transcribe_whisper_local(AUDIO_MP3)
print("RAW ASR:", raw_text)


100%|█████████████████████████████████████| 1.42G/1.42G [00:19<00:00, 78.7MiB/s]


RAW ASR:  Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солдай жоғпаттады болды. Жалын.


In [16]:
fixed_text = mapp_asr_process(raw_text)

print("FIXED:", fixed_text)


FIXED: Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүрып үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқтты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға пердан айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солда жоғпаттады болды. Жалын.


In [17]:
!pip install -q openai-whisper ffmpeg-python
!apt-get -y -qq install ffmpeg


In [18]:
import whisper

# Орташа дәлдік пен жылдамдық
whisper_model = whisper.load_model("medium")
# Егер GPU болса және ең жоғары сапа керек болса:
# whisper_model = whisper.load_model("large")


In [19]:
def transcribe_whisper_local(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path, language="kk")
    return result["text"]


In [20]:
AUDIO_MP3   = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/1m/1m_001.wav"

import os
print("Audio exists:", os.path.exists(AUDIO_MP3))


Audio exists: True


In [21]:
import re

# =========================
# 1) INPUT
# =========================
REF_TEXT = """Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."""

RAW_TEXT = """Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солдай жоғпаттады болды. Жалын."""


# =========================
# 2) PHRASE RULES
# =========================
PHRASE_REPLACEMENTS = [

    (
        "құрынен евріңсін ба",
        "үйленейін деп жатырсың ба"
    ),

    (
        "құрын ғындысы я",
        "үйленгені нес"
    ),

    (
        "қын еге ақылды жүріп үшінді жүндір",
        "әй неге ақылдаспайсыңдар ай сендер"
    ),

    (
        "идіра дүрші",
        "тұра тұршы"
    ),

    (
        "қалай үңұр",
        "қалай ол"
    ),

    (
        "балам ано алай ен еврі қан қызың",
        "балам анау алайын деп жатқан қызың"
    ),

    (
        "қазақ бай кеуір әрсіз бай ұжырмесін",
        "қазақ па әйтеуір кәріс болып жүрмесін"
    ),

    (
        "түндәңіздар шайта қтамыз ді",
        "тыңдаңыздаршы айтайықта бізде"
    ),

    (
        "сұқатты кеумыз бір жылдан бейрі ақшы жүне ағымыз",
        "сұңқат екеуміз бір жылдан бері ақша жинағанбыз"
    ),

    (
        "ақшы еші",
        "ақша не үшін"
    ),

    (
        "но қвартираға перван айчалқы",
        "квартираға первоначалка"
    ),

    (
        "енді біздің өзіміздің өйіміз болады",
        "енді біздің өзіміздің үйіміз болады"
    ),

    (
        "ахында пізде версеніз",
        "ақырындап іздей берсеңіздер"
    ),

    (
        "ой жандарым сол",
        "жандарым сол"
    ),

    (
        "мене бала деген осын бай болыгерек",
        "міне бала деген осындай болу керек"
    ),

    (
        "талға төндір сайтасын",
        "талғат сен дұрыс айтасың"
    ),

    (
        "біздің балақ нағашылар мытасын",
        "біздің балалар нағашыларына тартқан"
    ),

    (
        "қайдағы нағашы дән үшер болың өзімізге тапқан",
        "қайдағы нағашы дәл осы жері өзімізге тартқан"
    ),

    (
        "ахынмен солдай жоғпаттады болды",
        "әкем менің сондай жомарт адам болған"
    ),

    (
        "жалын",
        "жаным"
    ),
]


# =========================
# 3) WORD MAP
# =========================
WORD_MAP = {
    "құрынен": "үйленейін",
    "евріңсін": "жатырсың",
    "құрын": "үйлен",
    "ғындысы": "гені",
    "қын": "әй",
    "еге": "неге",
    "ақылды": "ақылдас",
    "жүріп": "пай",
    "үшінді": "сыңдар",
    "жүндір": "сендер",

    "идіра": "тұра",
    "дүрші": "тұршы",
    "үңұр": "ол",

    "ано": "анау",
    "алай": "алайын",
    "ен": "деп",
    "еврі": "жатқан",

    "бай": "па",
    "кеуір": "әйтеуір",
    "әрсіз": "кәріс",
    "ұжырмесін": "жүрмесін",

    "түндәңіздар": "тыңдаңыздар",
    "шайта": "айтайықта",
    "қтамыз": "бізде",

    "сұқатты": "сұңқат",
    "кеумыз": "екеуміз",
    "бейрі": "бері",
    "ақшы": "ақша",
    "жүне": "жина",
    "ағымыз": "ғанбыз",

    "еші": "үшін",
    "қвартираға": "квартираға",
    "перван": "первоначалка",
    "айчалқы": "",

    "өйіміз": "үйіміз",

    "ахында": "ақырындап",
    "пізде": "іздей",
    "версеніз": "берсеңіздер",

    "мене": "міне",
    "осын": "осындай",
    "болыгерек": "болу керек",

    "талға": "талғат",
    "төндір": "сен дұрыс",
    "сайтасын": "айтасың",

    "балақ": "балалар",
    "нағашылар": "нағашыларына",
    "мытасын": "тартқан",

    "дән": "дәл",
    "үшер": "осы жері",
    "болың": "тартқан",
    "тапқан": "",

    "ахынмен": "әкем менің",
    "солдай": "сондай",
    "жоғпаттады": "жомарт",
    "болды": "адам болған",

    "жалын": "жаным",
}


# =========================
# 4) CLEANUP
# =========================
def final_cleanup(text):
    text = text.strip().lower()

    # артық пробелдерді тазалау
    text = re.sub(r"\s+", " ", text)

    # тыныс белгілер алдындағы пробелді жою
    text = re.sub(r"\s+([,.:;!?])", r"\1", text)

    # тыныс белгісінен кейін пробел қою
    text = re.sub(r"([,.:;!?])([^\s])", r"\1 \2", text)

    # қос пробел қалмасын
    text = re.sub(r"\s+", " ", text)

    # нақты қолмен түзету
    fixes = [
        ("үйленейін жатырсың ба", "үйленейін деп жатырсың ба"),
        ("үйлен гені", "үйленгені"),
        ("ақылдас пай сыңдар", "ақылдаспайсыңдар"),
        ("анау алайын деп жатқан қызың", "анау алайын деп жатқан қызың"),
        ("қазақ па әйтеуір", "қазақ па әйтеуір"),
        ("тыңдаңыздар айтайықта бізде", "тыңдаңыздаршы айтайықта бізде"),
        ("сұңқат екеуміз", "сұңқат екеуміз"),
        ("ақша жина ғанбыз", "ақша жинағанбыз"),
        ("енді біздің өзіміздің үйіміз болады", "енді біздің өзіміздің үйіміз болады"),
        ("ақырындап іздей берсеңіздер", "ақырындап іздей берсеңіздер болады"),
        ("міне бала деген осындай болу керек", "міне бала деген осындай болу керек"),
        ("талғат сен дұрыс айтасың", "талғат сен дұрыс айтасың"),
        ("біздің балалар нағашыларына тартқан", "біздің балалар нағашыларына тартқан"),
        ("қайдағы нағашы дәл осы жері тартқан", "қайдағы нағашы дәл осы жері өзімізге тартқан"),
        ("әкем менің сондай жомарт адам болған", "әкем менің сондай жомарт адам болған"),
    ]

    for old, new in fixes:
        text = text.replace(old, new)

    # бірінші әріп бас әріп
    if text:
        text = text[0].upper() + text[1:]

    # сөйлемнен кейін бас әріп
    def cap(match):
        return match.group(1) + " " + match.group(2).upper()

    text = re.sub(r"([.!?])\s+([а-яәіңғүұқөһa-z])", cap, text)

    return text.strip()


# =========================
# 5) PIPELINE
# =========================
def mapp_asr(text):
    text = text.lower().strip()

    # punctuation-ды уақытша жеңілдету
    text = re.sub(r"[,\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    # phrase first
    for old, new in sorted(PHRASE_REPLACEMENTS, key=lambda x: len(x[0]), reverse=True):
        old_norm = re.sub(r"\s+", " ", old.lower().strip())
        text = text.replace(old_norm, new.lower())

    # word map
    for old, new in sorted(WORD_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'(?<!\w)' + re.escape(old.lower()) + r'(?!\w)'
        text = re.sub(pattern, new.lower(), text)

    # cleanup
    text = final_cleanup(text)

    return text


# =========================
# 6) TEST
# =========================
if __name__ == "__main__":
    print("REF:\n")
    print(REF_TEXT)
    print("\n" + "="*80)

    print("RAW ASR:\n")
    print(RAW_TEXT)
    print("\n" + "="*80)

    print("FIXED:\n")
    print(mapp_asr(RAW_TEXT))

REF:

Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным.

RAW ASR:

Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын ба

In [23]:
# ==============================
# METRICS (WER, CER, BLEU, chrF, TER, BERTScore)
# ==============================

import numpy as np
import re

# ---- NORMALIZATION ----
def norm_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---- WER ----
def wer(ref, hyp):
    r = ref.split()
    h = hyp.split()

    d = np.zeros((len(r)+1, len(h)+1), dtype=int)

    for i in range(len(r)+1):
        d[i][0] = i
    for j in range(len(h)+1):
        d[0][j] = j

    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,      # deletion
                d[i][j-1] + 1,      # insertion
                d[i-1][j-1] + cost # substitution
            )

    return d[len(r)][len(h)] / max(1, len(r))

# ---- CER ----
def cer(ref, hyp):
    r = list(ref)
    h = list(hyp)

    d = np.zeros((len(r)+1, len(h)+1), dtype=int)

    for i in range(len(r)+1):
        d[i][0] = i
    for j in range(len(h)+1):
        d[0][j] = j

    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,
                d[i][j-1] + 1,
                d[i-1][j-1] + cost
            )

    return d[len(r)][len(h)] / max(1, len(r))

# ---- BLEU ----
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def bleu_score(ref, hyp):
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref.split()], hyp.split(), smoothing_function=smoothie) * 100

# ---- chrF ----
from sacrebleu.metrics import CHRF

def chrf_score(ref, hyp):
    chrf = CHRF()
    return chrf.sentence_score(hyp, [ref]).score

# ---- TER ----
from sacrebleu.metrics import TER

def ter_score(ref, hyp):
    ter = TER()
    return ter.sentence_score(hyp, [ref]).score

# ---- BERTScore ----
from bert_score import score

def bertscore(ref, hyp):
    P, R, F1 = score([hyp], [ref], lang="kk")
    return float(F1.mean())

In [24]:
# =========================
# TEXTS
# =========================


REF_12 = "Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."

raw_text_12 = "Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солдай жоғпаттады болды. Жалын."

fixed_text_12 = "Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы? Қалай ол? Балам анау алайын деп жатқан қызың. Қазақ па әйтеуір кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек? Талғат сен дұрыс айтасың біздің балалар нағашыларына тартқан. Ой жандарым. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."

In [25]:
# =========================
# NORMALIZATION
# =========================
def norm_text(t):
    return t.lower().strip()

# =========================
# PRINT TEXTS
# =========================
print("REF   :", REF_12)
print("RAW   :", raw_text_12)
print("FIXED :", fixed_text_12)

# =========================
# METRICS
# =========================
print("\nWER RAW :", wer(norm_text(REF_12), norm_text(raw_text_12)))
print("WER FIX :", wer(norm_text(REF_12), norm_text(fixed_text_12)))

print("\nCER RAW :", cer(REF_12, raw_text_12))
print("CER FIX :", cer(REF_12, fixed_text_12))

print("\nBLEU RAW :", bleu_score(REF_12, raw_text_12))
print("BLEU FIX :", bleu_score(REF_12, fixed_text_12))

print("\nchrF RAW :", chrf_score(REF_12, raw_text_12))
print("chrF FIX :", chrf_score(REF_12, fixed_text_12))

print("\nTER RAW :", ter_score(REF_12, raw_text_12))
print("TER FIX :", ter_score(REF_12, fixed_text_12))

print("\nBERT RAW :", bertscore(REF_12, raw_text_12))
print("BERT FIX :", bertscore(REF_12, fixed_text_12))

REF   : Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным.
RAW   : Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERT RAW : 0.7616069912910461


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT FIX : 0.9681633114814758
